<a href="https://colab.research.google.com/github/yeshaa23/Project-A-Kelompok-4-Pertamina-PBAGenap/blob/main/Notebook/2b_Preprocessing_Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing Khusus BERT (Minimal Cleaning)

## Install dan Import Library

In [1]:
!pip install transformers torch
!pip install tqdm

In [2]:
import pandas as pd
import re
import string
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from tqdm import tqdm
from IPython.display import display

tqdm.pandas()
print("[→] Library siap!")

[→] Library siap!


## Load data hasil scraping

In [3]:
# Load Data Hasil Scraping
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
file_path = '/content/drive/MyDrive/ProjectA-PBA/hasil_scraping_berita.csv'
# Memuat data dari file CSV
df = pd.read_csv(file_path)
# Tampilkan beberapa baris pertama untuk memastikan data dimuat dengan benar
df.head()

Mounted at /content/drive


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit,Tanggal
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Netral,money.kompas.com,2017-10-03 00:00:00
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positif,money.kompas.com,2016-06-06 00:00:00
2,https://money.kompas.com/read/2017/02/03/12195...,"Dua Pucuk Pimpinan Pertamina Dicopot, Yenni An...","JAKARTA, KOMPAS.com — Pasca-pencopotan dua puc...",success,Migas,Positif,money.kompas.com,2017-02-03 00:00:00
3,https://money.kompas.com/read/2016/05/27/20434...,Ini yang Dilakukan PTKAM untuk Efisiensi 'Oil ...,"JAKARTA, KOMPAS.com - PT Pertamina (persero) t...",success,Gangguan Operasional,Positif,money.kompas.com,2016-05-27 00:00:00
4,https://money.kompas.com/read/2017/03/24/11200...,Ini Tugas Pertama Adiatma Sardjito Sebagai Jub...,"JAKARTA, KOMPAS.com - Adiatma Sardjito telah r...",success,Migas,Positif,money.kompas.com,2017-03-24 00:00:00


## Preprocessing sederhana

In [4]:
# ============================================================
# MEMUAT DATA BERITA PERTAMINA & PREPROCESSING DASAR
# ============================================================
print("=" * 60)
print("  PREPROCESSING BERITA PERTAMINA")
print("=" * 60)

# Memuat dataset hasil scraping
df = pd.read_csv(file_path, encoding="utf-8")  # bisa df_raw juga kalau mau tetap
print(f" Data dimuat: {len(df)} baris, {df.shape[1]} kolom")
print(f"\nKolom yang tersedia: {list(df.columns)}")

# ============================================================
# Mapping Sentimen ke English
# ============================================================
sentiment_map = {'Positif':'Positive', 'Negatif':'Negative', 'Netral':'Neutral'}
df['Sentimen'] = (
    df['Sentimen']
    .astype(str)
    .str.strip()
    .str.capitalize()
    .map(sentiment_map)
    .fillna(df['Sentimen'])
)

# ============================================================
# Normalisasi kolom Penerbit
# ============================================================
# Ubah semua penerbit menjadi Title Case + hapus spasi berlebih
df['Penerbit'] = df['Penerbit'].str.title().str.strip()

# Fungsi normalisasi publisher
def normalize_publisher(name):
    if not isinstance(name, str):
        return name

    name = name.strip().lower()  # lowercase + hapus spasi

    # Mapping domain/keyword
    if 'detik' in name:
        return 'Detik'
    elif 'kompas' in name:
        return 'Kompas'
    elif 'antara' in name:
        return 'Antara News'
    elif 'tempo' in name:
        return 'Tempo'
    elif 'kontan' in name:
        return 'Kontan'
    elif 'bisnis' in name:
        return 'Bisnis'
    else:
        return name.title()  # default: title case nama asli

# Terapkan ke kolom Penerbit
df['Penerbit'] = df['Penerbit'].apply(normalize_publisher)

display(df.head(10))

  PREPROCESSING BERITA PERTAMINA
 Data dimuat: 1883 baris, 8 kolom

Kolom yang tersedia: ['Link', 'Judul', 'Isi Berita', 'Status', 'Tag', 'Sentimen', 'Penerbit', 'Tanggal']


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit,Tanggal
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Neutral,Kompas,2017-10-03 00:00:00
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positive,Kompas,2016-06-06 00:00:00
2,https://money.kompas.com/read/2017/02/03/12195...,"Dua Pucuk Pimpinan Pertamina Dicopot, Yenni An...","JAKARTA, KOMPAS.com — Pasca-pencopotan dua puc...",success,Migas,Positive,Kompas,2017-02-03 00:00:00
3,https://money.kompas.com/read/2016/05/27/20434...,Ini yang Dilakukan PTKAM untuk Efisiensi 'Oil ...,"JAKARTA, KOMPAS.com - PT Pertamina (persero) t...",success,Gangguan Operasional,Positive,Kompas,2016-05-27 00:00:00
4,https://money.kompas.com/read/2017/03/24/11200...,Ini Tugas Pertama Adiatma Sardjito Sebagai Jub...,"JAKARTA, KOMPAS.com - Adiatma Sardjito telah r...",success,Migas,Positive,Kompas,2017-03-24 00:00:00
5,https://money.kompas.com/read/2017/01/16/15300...,"Akhirnya, Proyek Kilang Tuban Gunakan Lahan KL...","JAKARTA, KOMPAS.com - PT Pertamina (Persero) a...",success,Migas,Positive,Kompas,2017-01-16 00:00:00
6,https://nasional.kompas.com/read/2017/11/03/15...,"Jadi Tersangka Dana Pensiun Pertamina, Edward ...","JAKARTA, KOMPAS.com - Penyidik Jaksa Agung Mud...",success,Hukum,Negative,Kompas,2017-11-03 00:00:00
7,https://nasional.kompas.com/read/2017/08/16/16...,Viral Video Pertamax dan Pertalite Berwarna Sa...,"JAKARTA, KOMPAS.com - Seorang warganet bernama...",success,BBM,Neutral,Kompas,2017-08-16 00:00:00
8,https://otomotif.kompas.com/read/2017/04/21/13...,Spesial Buat Wanita di SPBU Pertamina,"Jakarta, KompasOtomotif – PT Pertamina (Perser...",success,BBM,Neutral,Kompas,2017-04-21 00:00:00
9,https://money.kompas.com/image/2017/08/03/0900...,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",Pelepasan Indukan Tuntong Laut oleh tim konser...,success,Bisnis,Positive,Kompas,2017-08-03 00:00:00


In [5]:
# ============================================================
# Definisi Fungsi Preprocessing Khusus BERT (Minimal Cleaning)
# ============================================================
def preprocess_for_bert(text):
    if not isinstance(text, str):
        return ""

    # 1. Case Folding: semua huruf kecil
    text = text.lower()

    # 2. Menghapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 3. Menghapus Mention (@user) dan Hashtag (#)
    text = re.sub(r'@\w+|#\w+', '', text)

    # 4. Menghapus Angka
    text = re.sub(r'\d+', '', text)

    # 5. Menghapus Tanda Baca
    import string
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 6. Menghapus Whitespace berlebih (spasi ganda, tab, newline)
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. Menghapus Boilerplate Berita yang tidak relevan (case-insensitive)
    boilerplate_patterns = [
        r'kompas\.?com', r'kompascom', r'detik\.com', r'tempo\.co',
        r'tribunnews\.com', r'kontan\.co\.id', r'bisnis\.com',
        r'com', r'co', r'id', r'news', r'finance', r'money', r'otomotif',
        r'jakarta', r'indonesia', r'bojonegoro', r'surabaya',
        r'pt', r'persero', r'tbk'
    ]
    for pattern in boilerplate_patterns:
        text = re.sub(r'\b' + pattern + r'\b', ' ', text, flags=re.IGNORECASE)

    # 8. Bersihkan lagi whitespace setelah boilerplate
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [6]:
# ============================================================
# TERAPKAN KE SETIAP ARTIKEL
# ============================================================
print("[→] Menjalankan preprocessing teks untuk BERT...")
df['content_cleaned_bert'] = df['Isi Berita'].progress_apply(preprocess_for_bert)

# Preview hasil
display(df[['Judul','Isi Berita','content_cleaned_bert']].head())

[→] Menjalankan preprocessing teks untuk BERT...


100%|██████████| 1883/1883 [00:07<00:00, 248.66it/s]


,Judul,Isi Berita,content_cleaned_bert
0,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,diskusi mengenai csr di industri pelumas berta...
1,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",anak perusahaan pertamina yakni pertamina lubr...
2,"Dua Pucuk Pimpinan Pertamina Dicopot, Yenni An...","JAKARTA, KOMPAS.com — Pasca-pencopotan dua puc...",— pascapencopotan dua pucuk pimpinan pertamina...
3,Ini yang Dilakukan PTKAM untuk Efisiensi 'Oil ...,"JAKARTA, KOMPAS.com - PT Pertamina (persero) t...",pertamina telah membentuk sebuah program untuk...
4,Ini Tugas Pertama Adiatma Sardjito Sebagai Jub...,"JAKARTA, KOMPAS.com - Adiatma Sardjito telah r...",adiatma sardjito telah resmi diangkat sebagai ...


## Simpan hasil preprocessing

In [7]:
# ============================================================
# SIMPAN DATA FINAL
# ============================================================
OUTPUT_FILE = "hasil_preprocessing_bert.csv"
OUTPUT_PATH = f"/content/drive/MyDrive/ProjectA-PBA/{OUTPUT_FILE}"

# Kolom final + kolom asli
columns_to_save = ['Link', 'Judul', 'Isi Berita', 'Status', 'Tag', 'Sentimen', 'Penerbit', 'content_cleaned_bert']

df[columns_to_save].to_csv(OUTPUT_PATH, index=False)
print(f"Dataset final berhasil disimpan: {OUTPUT_PATH}")

Dataset final berhasil disimpan: /content/drive/MyDrive/ProjectA-PBA/hasil_preprocessing_bert.csv
